# How LLM Architecture Evolved, 2017–2026: A Short List of Substitutions

The runnable code of [How LLM Architecture Evolved, 2017–2026: A Short List of Substitutions](https://residual-stream.lvtan71.workers.dev/posts/llm-architecture-evolution/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/llm-architecture-evolution.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "llm-architecture-evolution"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Memory at inference: GQA and MLA

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/llm-architecture-evolution/#memory-at-inference-gqa-and-mla)

In [ ]:
# What the attention substitutions save: the KV cache per generated token
# ("The KV Cache"), in 16-bit numbers, for published configurations, and the
# attention weights per layer. MHA keeps a key and a value per head; GQA one
# per group of query heads; MLA a compressed vector per token plus a small
# rotated key (DeepSeek-AI, 2024).
def kv_bytes(layers, per_layer_elements, bytes_per=2):
    return layers * per_layer_elements * bytes_per

configs = [
    # name, layers, model width, query heads, key/value heads, head size
    ("GPT-3 175B (MHA)", 96, 12288, 96, 96, 128),
    ("Llama 2 70B (GQA, 8 groups)", 80, 8192, 64, 8, 128),
    ("Llama 3 8B (GQA, 8 groups)", 32, 4096, 32, 8, 128),
]
for name, L, D, nq, nkv, h in configs:
    cache = kv_bytes(L, 2 * nkv * h)
    attn = D * nq * h * 2 + D * nkv * h * 2                     # Q and output, then K and V
    print(f"{name:30s} KV cache {cache / 1024:7.0f} KiB per token; attention weights per layer "
          f"{attn / 1e6:6.1f}M (with full MHA: {4 * D * nq * h / 1e6:6.1f}M)")

# DeepSeek-V2: 60 layers, 128 heads of 128, KV compression 512, rotated key 64.
L, nh, dh, dc, dr = 60, 128, 128, 512, 64
mla, mha = kv_bytes(L, dc + dr), kv_bytes(L, 2 * nh * dh)
print(f"{'DeepSeek-V2 (MLA)':30s} KV cache {mla / 1024:7.0f} KiB per token; the same heads with MHA: "
      f"{mha / 1024:,.0f} KiB, {mha / mla:.0f} times as much")

# Mixture of experts: parameters held vs parameters used per token.
for name, total, active in (("Mixtral 8x7B", 47e9, 13e9), ("DeepSeek-V2", 236e9, 21e9)):
    print(f"{name:30s} {total / 1e9:5.1f}B parameters, {active / 1e9:4.1f}B used per token "
          f"({active / total:.0%}): per-token FLOPs of a {active / 1e9:.0f}B model, memory of a {total / 1e9:.0f}B one")

The post printed:

```text
GPT-3 175B (MHA)               KV cache    4608 KiB per token; attention weights per layer  604.0M (with full MHA:  604.0M)
Llama 2 70B (GQA, 8 groups)    KV cache     320 KiB per token; attention weights per layer  151.0M (with full MHA:  268.4M)
Llama 3 8B (GQA, 8 groups)     KV cache     128 KiB per token; attention weights per layer   41.9M (with full MHA:   67.1M)
DeepSeek-V2 (MLA)              KV cache      68 KiB per token; the same heads with MHA: 3,840 KiB, 57 times as much
Mixtral 8x7B                    47.0B parameters, 13.0B used per token (28%): per-token FLOPs of a 13B model, memory of a 47B one
DeepSeek-V2                    236.0B parameters, 21.0B used per token (9%): per-token FLOPs of a 21B model, memory of a 236B one
```

## How big, on how much data: the scaling laws

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/llm-architecture-evolution/#how-big-on-how-much-data-the-scaling-laws)

In [ ]:
# Compute-optimal training from a fitted scaling law,
#     L(N, D) = E + A / N^alpha + B / D^beta,
# the loss of a model with N parameters trained on D tokens (here D is
# tokens, not width), with training compute C = 6 N D ("Backpropagation, by Hand"). Three
# versions of one fit: Hoffmann et al.'s (2022, "Chinchilla", approach 3) as
# printed in the paper, the same with the unrounded constants from the
# paper's source, and Besiroglu et al.'s (2024) refit of the same data.
FITS = {"Hoffmann, as printed": dict(E=1.69, A=406.4, B=410.7, alpha=0.34, beta=0.28),
        "Hoffmann, unrounded": dict(E=1.6934, A=406.4, B=410.7, alpha=0.3392, beta=0.2849),
        "Besiroglu refit": dict(E=1.8172, A=482.01, B=2085.43, alpha=0.3478, beta=0.3658)}

def loss(f, N, D):
    return f["E"] + f["A"] / N ** f["alpha"] + f["B"] / D ** f["beta"]

def optimal(f, C):
    """Minimizing L subject to 6ND = C. Setting dL/dN = 0 along the
    constraint gives alpha A / N^alpha = beta B / D^beta, so
    N* = G (C/6)^(beta/(alpha+beta)), G = (alpha A / (beta B))^(1/(alpha+beta))."""
    a, b = f["alpha"], f["beta"]
    G = (a * f["A"] / (b * f["B"])) ** (1 / (a + b))
    N = G * (C / 6) ** (b / (a + b))
    return N, C / 6 / N

for name, f in FITS.items():
    rows = []
    for C in (1e21, 1e23, 1e25):
        N, D = optimal(f, C)
        rows.append(f"{D / N:5.1f}")
    print(f"{name:21s} tokens per parameter at C = 1e21 / 1e23 / 1e25 FLOPs: {' / '.join(rows)}")

# Llama 3 8B: about 8 billion parameters, trained on about 15 trillion tokens.
N8, D8 = 8e9, 15e12
C = 6 * N8 * D8
print(f"\nLlama 3 8B: {D8 / N8:,.0f} tokens per parameter, C = {C:.1e} FLOPs")
for name, f in FITS.items():
    No, Do = optimal(f, C)
    target = loss(f, N8, D8)
    lo, hi = 1e18, C                                    # least compute whose optimal model reaches the 8B's loss
    for _ in range(100):
        mid = (lo * hi) ** 0.5
        lo, hi = (mid, hi) if loss(f, *optimal(f, mid)) > target else (lo, mid)
    Nm, _ = optimal(f, hi)
    print(f"  {name:21s} same compute: optimal {No / 1e9:3.0f}B on {Do / 1e12:4.1f}T tokens, loss {loss(f, No, Do):.3f} "
          f"vs the 8B's {target:.3f}; same loss: a {Nm / 1e9:.0f}B model with {hi / C:.2f}x the 8B's training compute")

# For the chart: loss against model size at the 8B's training compute.
for name, f in FITS.items():
    pts = [(n, loss(f, n * 1e9, C / 6 / (n * 1e9))) for n in (2, 4, 8, 16, 32, 64, 128, 256)]
    print(f"curve {name}: " + " ".join(f"{n}:{l:.4f}" for n, l in pts))

The post printed:

```text
Hoffmann, as printed  tokens per parameter at C = 1e21 / 1e23 / 1e25 FLOPs:  50.1 /  78.2 / 122.1
Hoffmann, unrounded   tokens per parameter at C = 1e21 / 1e23 / 1e25 FLOPs:  34.0 /  50.7 /  75.7
Besiroglu refit       tokens per parameter at C = 1e21 / 1e23 / 1e25 FLOPs:  21.6 /  19.2 /  17.1

Llama 3 8B: 1,875 tokens per parameter, C = 7.2e+23 FLOPs
  Hoffmann, as printed  same compute: optimal  36B on  3.4T tokens, loss 1.923 vs the 8B's 1.949; same loss: a 26B model with 0.50x the 8B's training compute
  Hoffmann, unrounded   same compute: optimal  45B on  2.7T tokens, loss 1.911 vs the 8B's 1.944; same loss: a 29B model with 0.40x the 8B's training compute
  Besiroglu refit       same compute: optimal  81B on  1.5T tokens, loss 1.968 vs the 8B's 2.022; same loss: a 34B model with 0.18x the 8B's training compute
curve Hoffmann, as printed: 2:2.0266 4:1.9801 8:1.9485 16:1.9299 32:1.9228 64:1.9264 128:1.9405 256:1.9654
curve Hoffmann, unrounded: 2:2.0266 4:1.9777 8:1.9435 16:1.9221 32:1.9119 64:1.9121 128:1.9223 256:1.9426
curve Besiroglu refit: 2:2.1169 4:2.0622 8:2.0221 16:1.9941 32:1.9766 64:1.9688 128:1.9703 256:1.9813
```